# W2: Casablanca validation preparation and audit (CPU)
Downloads the pinned Casablanca **validation** shards only, audits them, extracts the original WAVs and freezes the
evaluation manifest. Rules: `docs/casablanca_validation.md`. No model is loaded and nothing is scored.

**Before running:** a CPU runtime is enough. Drive needs about **9.5 GB free** (4.6 GB original Parquet + 4.6 GB extracted WAVs).
Casablanca **test** is never downloaded or read.

Build the code ZIP from the repository root on your computer, then upload it in the first cell:

    (git ls-files -co --exclude-standard; find .git -type f) | zip -q Arabic-Dialect-Identification-casablanca.zip -@

If the runtime disconnects, run all cells again: finished downloads are reused and extracted WAVs are checked, not rewritten.

In [ ]:
from google.colab import drive, files
from datetime import datetime, timezone
from pathlib import Path
import io
import json
import subprocess
import sys
import zipfile

drive.mount('/content/drive')
# Same data folder as the ADI-20 notebooks; Casablanca goes under raw/casablanca/ and audio/casablanca/.
DATA_ROOT = Path('/content/drive/MyDrive/Arabic-Dialect-Identification-data')
CODE_ROOT = Path('/content/casablanca-code')
LOG_DIR = DATA_ROOT / 'reports' / 'casablanca'
for folder in (DATA_ROOT, LOG_DIR):
    folder.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()  # select Arabic-Dialect-Identification-casablanca.zip
archives = [name for name in uploaded if name.endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Upload exactly one code ZIP')
with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
    archive.extractall(CODE_ROOT)
if not (CODE_ROOT / 'src/data/prepare_casablanca.py').is_file():
    raise ValueError('This ZIP does not contain src/data/prepare_casablanca.py; rebuild it from the repository root')
print('Code:', CODE_ROOT)
print('Data:', DATA_ROOT)


def run_logged(command, name):
    """Stream a command's output and keep a copy on Drive."""
    log_path = LOG_DIR / f"{name}-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}.log"
    print('Log:', log_path, flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, cwd=CODE_ROOT)
    with log_path.open('a') as log:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
    return process.wait()

## Install and run the offline checks
Synthetic tests only; nothing is downloaded here.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE_ROOT / 'requirements-data.txt')], check=True)
# discover -s tests imports the file directly; Colab has an unrelated installed `tests` package.
code = run_logged([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_prepare_casablanca.py',
                   '-v'], 'tests')
assert code == 0, 'Offline checks failed; stop here and report the log'

## Download, audit and freeze
Exit code 0 = no audio errors, 2 = finished with audio errors kept in the denominator (review `exclusions.jsonl`),
1 = stopped for review (for example a row count different from the published 6,819).

In [ ]:
exit_code = run_logged([sys.executable, '-u', 'src/data/prepare_casablanca.py', '--data-root', str(DATA_ROOT)],
                       'prepare-casablanca')
print('Exit code:', exit_code)
assert exit_code in (0, 2), 'Preparation stopped; read the log above before rerunning'

## Small preprocessing check
Runs the 10 longest and 40 seeded random clips through the acoustic model's audio loader
(mean downmix, 16 kHz sinc resample, full clip). No encoder is loaded.

In [ ]:
try:
    import torchaudio  # noqa: F401
except ImportError:
    import torch
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    f"torchaudio=={torch.__version__.split('+')[0]}"], check=True)
code = run_logged([sys.executable, 'src/data/check_casablanca_preprocessing.py', '--data-root', str(DATA_ROOT)],
                  'preprocessing-check')
print('Exit code:', code)

## Summary and results ZIP
Download `casablanca-results.zip` and send it back. It holds only manifests, hashes and audit reports, no audio.
A copy is also saved on Drive.

In [ ]:
audit = json.loads((CODE_ROOT / 'reports/casablanca_validation_audit/audit.json').read_text())
summary = audit['summary']
print('Status:', audit['status'], '| split version:', audit['split_version'])
print('Clips:', summary['rows'], '| ok:', summary['ok'], '| audio errors:', summary['audio_errors'])
print('Per label:', audit['decoded_counts'])
print('Hours:', round(summary['duration_hours'], 2), '| duration (s):', summary['duration_sec'])
print('Sample rates:', summary['sample_rate_counts'], '| channels:', summary['channel_counts'])
print('Exact duplicate groups:', audit['exact_duplicate_groups'], '| label conflicts:',
      audit['exact_duplicate_label_conflicts'])
print('ADI-20 exact overlap findings:', audit['adi20_overlap_findings'])
print('Duration mismatches > 10 ms:', audit['duration_mismatches_over_tolerance'])

results = [CODE_ROOT / 'data/manifests' / name for name in
           ('casablanca_validation.jsonl', 'casablanca_validation.metadata.json', 'casablanca_validation_freeze.json')]
results += sorted((CODE_ROOT / 'reports/casablanca_validation_audit').glob('*'))
results += sorted(LOG_DIR.glob('*.log'))
zip_path = LOG_DIR / 'casablanca-results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in results:
        name = path.relative_to(CODE_ROOT) if path.is_relative_to(CODE_ROOT) else Path('logs') / path.name
        archive.write(path, str(name))
print('Saved:', zip_path)
files.download(str(zip_path))